[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/urban/blob/feat/denue-colab-workspace/notebooks/denue/00_COLAB_START_HERE.ipynb)

# Urban · DENUE histórico · Colab

Notebook de arranque para clonar el repositorio, instalar el entorno y ejecutar el pipeline reproducible de DENUE.

**No requiere volver a subir el ZIP manualmente** una vez que el archivo auditado esté en la rama `data/denue-historico`.

In [ ]:
REPO = 'https://github.com/Murua-Alvaro/urban.git'
BRANCH = 'feat/denue-colab-workspace'
WORKDIR = '/content/urban'
USE_DRIVE_CACHE = False
STAGE = 'eda'  # load | validate | clean | eda | spatial | features | model | all
WITH_MORAN = False

print({'branch': BRANCH, 'stage': STAGE, 'drive_cache': USE_DRIVE_CACHE})

In [ ]:
import os, subprocess, pathlib, shutil

if pathlib.Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, WORKDIR], check=True)
os.chdir(WORKDIR)
print('Repo listo en', os.getcwd())

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Dependencias instaladas')

In [ ]:
from pathlib import Path

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    DENUE_ROOT = Path('/content/drive/MyDrive/urban-denue-cache')
else:
    DENUE_ROOT = Path('/content/urban-denue')

DENUE_ROOT.mkdir(parents=True, exist_ok=True)
print('DENUE_ROOT =', DENUE_ROOT)

In [ ]:
import subprocess, sys
cmd = [sys.executable, 'scripts/denue/colab_entrypoint.py', '--stage', STAGE, '--root', str(DENUE_ROOT)]
if WITH_MORAN:
    cmd.append('--with-moran')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## Inspección rápida

Las siguientes celdas buscan salidas Parquet/CSV generadas por el pipeline y permiten empezar el análisis interactivo.

In [ ]:
from pathlib import Path
outputs = sorted(DENUE_ROOT.rglob('*.parquet')) + sorted(DENUE_ROOT.rglob('*.csv'))
print(f'{len(outputs)} archivos analíticos encontrados')
for p in outputs[:40]:
    print(p.relative_to(DENUE_ROOT))

In [ ]:
import pandas as pd

candidates = [p for p in DENUE_ROOT.rglob('*.parquet') if 'municip' in p.name.lower() or 'state' in p.name.lower()]
if candidates:
    sample = pd.read_parquet(candidates[0])
    display(sample.head(20))
else:
    print('Ejecuta STAGE = "eda" o "all" para generar tablas de EDA.')

## Ejecución completa

Para correr validación, limpieza, EDA, análisis espacial, feature engineering **y modelado**, cambia `STAGE = 'all'` en la primera celda y elige **Entorno de ejecución → Ejecutar todas**.

Si solo quieres llegar a los modelos, también puedes usar `STAGE = 'model'`.

Usa `USE_DRIVE_CACHE = True` si quieres que la caché y los resultados persistan entre sesiones de Colab.